# LSTM Baseline Model (L0)

## Objective
Establish a baseline for English-to-French translation using an LSTM encoder–decoder architecture without attention.

## Architecture
- Encoder: one LSTM layer
- Decoder: one LSTM layer
- English embedding dimension: 128
- French embedding dimension: 128
- Hidden dimension: 256
- Dropout: 0.2

## Training Configuration
- Optimizer: Adam
- Initial learning rate: 0.001
- Weight decay: 0.00001
- Batch size: 64
- Maximum epochs: 20
- Loss: Cross-entropy, ignoring French padding tokens
- Gradient clipping: maximum norm 1.0
- Scheduler: ReduceLROnPlateau

## Objective of the Experiment
Compare the LSTM baseline against the GRU baseline using validation loss, token accuracy, training time, parameter count, and eventually generated translation quality.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Import necessary libraries
import sys
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from tokenizers import Tokenizer

In [4]:
%cd /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation

/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation


In [5]:
%ls -la

total 34
drwx------ 6 root root 4096 Oct 12 09:24 ./
drwx------ 3 root root 4096 Oct 12 09:02 ../
drwx------ 2 root root 4096 Oct 12 10:32 data/
drwx------ 2 root root 4096 Oct 12 09:24 .git/
-rw------- 1 root root    0 Oct 12 09:28 .gitignore
drwx------ 2 root root 4096 Oct 12 09:28 notebooks/
-rw------- 1 root root 5120 Oct 12 09:33 README.md
drwx------ 2 root root 4096 Oct 12 09:28 src/
-rw------- 1 root root 5120 Oct 12 09:32 utils


In [6]:
print(Path.cwd())

/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation


In [8]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Source directory:", SRC_DIR)
print("Device:", device)

Project root: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation
Data directory: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/data
Source directory: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/src
Device: cpu


In [10]:
# Load processed data
tensor_path = PROCESSED_DIR / "translation_ids.pt"

data = torch.load(
    tensor_path,
    map_location="cpu",
    weights_only=True
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]


# Load tokenizers
en_tokenizer = Tokenizer.from_file(
    str(PROCESSED_DIR / "english_tokenizer.json")
)

fr_tokenizer = Tokenizer.from_file(
    str(PROCESSED_DIR / "french_tokenizer.json")
)


# Load preprocessing configuration
with open(
    PROCESSED_DIR / "preprocessing_config.json",
    "r",
    encoding="utf-8"
) as file:
    config = json.load(file)


print("Data loaded successfully.")
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("English vocabulary:", en_tokenizer.get_vocab_size())
print("French vocabulary:", fr_tokenizer.get_vocab_size())
print("Configuration:", config)

Data loaded successfully.
Training: torch.Size([133704, 20]) torch.Size([133704, 22])
Validation: torch.Size([16713, 20]) torch.Size([16713, 22])
Test: torch.Size([16713, 20]) torch.Size([16713, 22])
English vocabulary: 14493
French vocabulary: 23539
Configuration: {'english_max_length': 20, 'french_max_length': 22, 'english_vocab_size': 14493, 'french_vocab_size': 23539, 'pad_token': '[PAD]', 'unk_token': '[UNK]', 'bos_token': '[BOS]', 'eos_token': '[EOS]'}


In [11]:
# Retrieve special token IDs
en_pad_id = en_tokenizer.token_to_id("[PAD]")
fr_pad_id = fr_tokenizer.token_to_id("[PAD]")

en_unk_id = en_tokenizer.token_to_id("[UNK]")
fr_unk_id = fr_tokenizer.token_to_id("[UNK]")

en_bos_id = en_tokenizer.token_to_id("[BOS]")
fr_bos_id = fr_tokenizer.token_to_id("[BOS]")

en_eos_id = en_tokenizer.token_to_id("[EOS]")
fr_eos_id = fr_tokenizer.token_to_id("[EOS]")


# Create datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)


# Create data loaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Import shared utilities
from src.utils import train_model, plot_training_history, evaluate_model


# Verify
print("English PAD ID:", en_pad_id)
print("French PAD ID:", fr_pad_id)
print("French BOS ID:", fr_bos_id)
print("French EOS ID:", fr_eos_id)

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

source_batch, target_batch = next(iter(train_loader))

print("\nSource batch shape:", source_batch.shape)
print("Target batch shape:", target_batch.shape)

English PAD ID: 1
French PAD ID: 1
French BOS ID: 2
French EOS ID: 3

Training batches: 2090
Validation batches: 262
Test batches: 262

Source batch shape: torch.Size([64, 20])
Target batch shape: torch.Size([64, 22])


In [12]:
# Encoder class

class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        padding_idx
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, (hidden, cell) = self.lstm(packed)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=input_ids.shape[1]
        )

        return outputs, (hidden, cell)

In [13]:
# Decoder class

class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        padding_idx,
        dropout_rate=0.2
    ):
        super().__init__()

        # Convert French token IDs into vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )

        # Process the French sequence
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # Dropout before the output projection
        self.dropout = nn.Dropout(dropout_rate)

        # Convert each LSTM output into vocabulary scores
        self.fc = nn.Linear(
            in_features=hidden_dim,
            out_features=vocab_size,
        )

    def forward(self, input_ids, state):
        # Unpack the encoder's hidden and cell states
        hidden, cells = state

        # (batch_size, target_length, embedding_dim)
        embedded = self.embedding(input_ids)

        # Process the sequence using both initial states
        outputs, (hidden, cells) = self.lstm(
            embedded,
            (hidden, cells)
        )

        # Apply dropout to LSTM outputs
        outputs = self.dropout(outputs)

        # (batch_size, target_length, vocab_size)
        logits = self.fc(outputs)

        return logits, (hidden, cells)

In [14]:
# Seq2Seq class

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source_ids, source_lengths, target_ids):
        # Encode the English sentences
        encoder_outputs, (encoder_hidden, encoder_cell) = self.encoder(
            source_ids,
            source_lengths,
        )

        # Initialize the decoder with both encoder states
        logits, (decoder_hidden, decoder_cell) = self.decoder(
            target_ids,
            (encoder_hidden, encoder_cell),
        )

        return logits